# Small Models, Real Constraints — Runner

This notebook runs the full pipeline end to end on **Kaggle or Colab** (both CPU-only, which is the whole point of the project).

**Order:** setup → smoke test → data → sparsity report → baselines → Track A → Track B → analysis → figures.

> Tip: run the smoke test first. If it prints `SMOKE TEST PASSED`, everything else will work.


## 1. Get the code

Detects the environment and pulls the repo. On Colab it clones from GitHub. On Kaggle, either add the repo as a *Dataset/Utility Script* or clone if internet is enabled.

In [ ]:
import os, sys, subprocess

GITHUB_URL = "https://github.com/<your-username>/small-models-real-constraints"   # <-- set this to your repo
REPO_DIR = "small-models-real-constraints"

def sh(cmd):
    print("$", cmd)
    print(subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout)

ON_COLAB = "google.colab" in sys.modules
ON_KAGGLE = os.path.exists("/kaggle")
print("Colab:", ON_COLAB, "| Kaggle:", ON_KAGGLE)

# Strategy:
#  - If the repo folder already exists (e.g. uploaded to Kaggle), just use it.
#  - Otherwise clone it from GitHub (needs internet enabled on Kaggle).
if os.path.isdir(REPO_DIR):
    print("Repo folder already present.")
elif os.path.isdir("/kaggle/input"):
    # Look for the repo among attached Kaggle datasets/utility scripts.
    found = None
    for root, dirs, files in os.walk("/kaggle/input"):
        if "requirements.txt" in files and os.path.isdir(os.path.join(root, "src")):
            found = root; break
    if found:
        sh(f"cp -r '{found}' {REPO_DIR}")
        print("Copied repo from Kaggle input:", found)
    else:
        sh(f"git clone {GITHUB_URL} {REPO_DIR}")
else:
    sh(f"git clone {GITHUB_URL} {REPO_DIR}")

assert os.path.isdir(REPO_DIR), "Repo not found. Set GITHUB_URL or attach the repo as a Kaggle dataset."
os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())


## 2. Install dependencies

Most are preinstalled on Kaggle/Colab. This just fills any gaps (notably `datasets` and `psutil`).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
print("dependencies ready")


## 3. Smoke test (no downloads)

Runs the whole pipeline on tiny synthetic data. Confirms every stage imports and executes.

In [ ]:
!python scripts/smoke_test.py


## 4. Load AG News

The loader tries, in order: a local CSV in `data/`, then HuggingFace `datasets`, then a direct CSV download. On Kaggle you can also **attach the AG News dataset** and drop `train.csv` / `test.csv` into `data/` to run fully offline.


In [ ]:
from src import config, data
from src.sparsity import analyse, pretty

ds = data.load(prefer="auto")   # use prefer="local" if you placed CSVs in data/
Xtr, Xte, vec = ds.vectorise()
print("train:", Xtr.shape, "| test:", Xte.shape)
print(pretty(analyse(Xtr)))


## 5. Point 1 — sparsity / RAM savings report

In [ ]:
!python scripts/report_sparsity.py auto


## 6. Baselines (Week 1 deliverable)

Three tuned classical models. This is the slowest cell; on AG News expect a few minutes on CPU.

In [ ]:
!python scripts/run_baselines.py auto


## 7. Track A — pruning sweeps (point 2)

Feature selection (chi² / mutual information / L1), document-frequency filtering, and weight pruning.

> **Runtime note:** the `mutual_info` selector is the slow part on the full 20k vocabulary. If you want a quick pass first, comment it out in `src/config.py` (`FEATURE_SELECTORS`).

In [ ]:
!python scripts/run_track_a.py auto


## 8. Track B — simulated quantisation (point 4)

fp32 → int16/int8/int4 for all three models, including Naive Bayes' log-probability tables.

In [ ]:
!python scripts/run_track_b.py auto


## 9. Analysis — F1 vs efficiency index + Pareto (point 5)

In [ ]:
!python -m src.analysis


## 10. Figures

Generates all report figures, then displays them inline.

In [ ]:
!python scripts/make_figures.py

from IPython.display import Image, display
import glob, os
for png in sorted(glob.glob("figures/*.png")):
    print(os.path.basename(png))
    display(Image(filename=png))


## 11. Save / download outputs

Bundles `results/` and `figures/` into a zip you can download for the report and submission.

In [ ]:
import shutil, os
shutil.make_archive("smrc_outputs", "zip", ".", "results")
shutil.make_archive("smrc_figures", "zip", ".", "figures")
print("Wrote smrc_outputs.zip and smrc_figures.zip in", os.getcwd())

# On Colab, trigger a download:
try:
    from google.colab import files
    files.download("smrc_outputs.zip")
    files.download("smrc_figures.zip")
except Exception:
    print("On Kaggle: use the file browser on the right to download, or commit the notebook.")
